In [ ]:
!pip install -q transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

In [ ]:
english_df = pd.read_csv("/content/English_train_data.csv")
hindi_df = pd.read_csv("/content/final_dataset_hindi.csv")
bengali_df = pd.read_csv("/content/merged_bengali_dataset.csv")
hindi_df2 = pd.read_csv("/content/english_to_hindi.csv")
bengali_df2=pd.read_csv("/content/english_to_bengali.csv")

In [ ]:
print(english_df.head())
english_df.shape

                                            sentence    label
0  : If we are going to slow climate change, we m...    FAVOR
1  World leaders duped by manipulated global warm...  AGAINST
2  To be sure the American greens have greater am...     NONE
3  Dealing with global warming is an internationa...    FAVOR
4  : Concussion is like climate change It isn't r...  AGAINST


(12771, 2)

In [ ]:
print(hindi_df.head())
hindi_df.shape

                                                text
0  सूर्य ग्रहण: दिन में चार मिनट तक रहेगा अंधेरा,...
1  भारतीय महिला बैडमिंटन टीम ने यंग ब्रिगेड के सह...
2  नेपाल में बोधिचित्त वृक्षों की चिंता: 'सोने की...
3  आईपीएल के सुपर संडे में कई रिकॉर्ड बने, लेकिन ...
4  भारत ईरान ने मतभेद भुलाकर किया ये अहम समझौता, ...


(4600, 1)

In [ ]:
print(bengali_df.head())
bengali_df.shape

                                                text
0  মার্কিন মহাকাশ সংস্থা নাসা আয়োজিত বিশ্বের বৃহ...
1  বঙ্গবন্ধুর জন্ম শতবর্ষ উপলক্ষে সারাদেশে ১ কোটি...
2  জার্মানির আন্তর্জাতিক সম্প্রচার কেন্দ্র ডয়চে ভ...
3  অর্থমন্ত্রী আ হ ম মুস্তফা কামাল বলেছেন, পেঁয়াজ...
4  কাপ্তাই হ্রদের দূষণ দূর করতে অভিযান পরিচালনা ক...


(4600, 1)

In [ ]:
print(hindi_df2.head())
hindi_df2.shape

                                            sentence    label  \
0  : If we are going to slow climate change, we m...    FAVOR   
1  World leaders duped by manipulated global warm...  AGAINST   
2  To be sure the American greens have greater am...     NONE   
3  Dealing with global warming is an internationa...    FAVOR   
4  : Concussion is like climate change It isn't r...  AGAINST   

                                               hindi  
0  : यदि हम जलवायु परिवर्तन को धीमा करना चाहते है...  
1  ग्लोबल वार्मिंग डेटा में हेरफेर करके विश्व नेत...  
2  यह सुनिश्चित करने के लिए कि मैक्रोन की तरह अमे...  
3  ग्लोबल वार्मिंग से निपटना एक अंतरराष्ट्रीय जिम...  
4  : हिलाना जलवायु परिवर्तन की तरह है यह वास्तविक...  


(12771, 3)

In [ ]:
print(bengali_df2.head())
bengali_df2.shape

                                            sentence    label  \
0  : If we are going to slow climate change, we m...    FAVOR   
1  World leaders duped by manipulated global warm...  AGAINST   
2  To be sure the American greens have greater am...     NONE   
3  Dealing with global warming is an internationa...    FAVOR   
4  : Concussion is like climate change It isn't r...  AGAINST   

                                             bengali  
0  : আমরা যদি জলবায়ু পরিবর্তনের গতি কমাতে যাচ্ছি...  
1  বিশ্বনেতারা বিশ্ব উষ্ণায়নের তথ্য ব্যবহার করে ...  
2  আমেরিকান সবুজ শাক-সবুজদের শুধু কার্বন ট্যাক্স ...  
3  বৈশ্বিক উষ্ণতা মোকাবেলা একটি আন্তর্জাতিক দায়িত্ব  
4  : উপহাস জলবায়ু পরিবর্তনের মতো এটি বাস্তব নয় ...  


(12771, 3)

In [ ]:
english = english_df["sentence"].astype(str)

hindi = hindi_df["text"].astype(str)

bengali = bengali_df["text"].astype(str)

hindi2 = hindi_df2["hindi"].astype(str)

bengali2 = bengali_df2["bengali"].astype(str)

In [ ]:
all_text = pd.concat(
    [
        english,
        hindi,
        bengali,
        hindi2,
        bengali2
    ],
    ignore_index=True
)

all_text = all_text.dropna()

all_text = all_text.drop_duplicates()

all_text = all_text.str.strip()

all_text = all_text[all_text!=""]

print("Total sentences:",len(all_text))

Total sentences: 46984


In [ ]:
mlm_df = pd.DataFrame()

mlm_df["text"] = all_text

mlm_df.head()

,text
0,": If we are going to slow climate change, we m..."
1,World leaders duped by manipulated global warm...
2,To be sure the American greens have greater am...
3,Dealing with global warming is an internationa...
4,: Concussion is like climate change It isn't r...


In [ ]:
train_df, valid_df = train_test_split(
    mlm_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(len(train_df))
print(len(valid_df))

42285
4699


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [ ]:
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 7.75MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
MAX_LENGTH = 256

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/42285 [00:00<?, ? examples/s]

Map:   0%|          | 0/4699 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.remove_columns(["text"])

valid_dataset = valid_dataset.remove_columns(["text"])

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.12GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertForMaskedLM LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/IndicBERT_MLM",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_steps=159,

    fp16=True,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    report_to="none"

)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    data_collator=data_collator

)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.925438,1.812322
2,3.631086,1.761006
3,3.495540,1.769526
4,3.635320,1.721376


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.925438,1.812322
2,3.631086,1.761006
3,3.495540,1.769526
4,3.635320,1.721376
5,3.624839,1.720250


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].


TrainOutput(global_step=13215, training_loss=3.706384744402409, metrics={'train_runtime': 12777.1346, 'train_samples_per_second': 16.547, 'train_steps_per_second': 1.034, 'total_flos': 2.78953139902464e+16, 'train_loss': 3.706384744402409, 'epoch': 5.0})

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/Best_IndicBERT_MLM_NEW"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Successfully!
